In [19]:
import pandas as pd
import numpy as np
import re

In [2]:
df = pd.read_csv('../Sales-[5444].csv')

In [3]:
df.shape

(1000, 10)

In [4]:
df.head()

,Item_Name,details,Catatory,price,priceWithCurrency,available,text,sold,lastUpdated,itemLocation
0,Dior,Christian Dior Sauvage Men's EDP 3.4 oz Fragra...,Eau de Parfum,84.99,US $84.99/ea,10.0,More than 10 available / 116 sold,NaN,"May 24, 2024 10:03:04 PDT","Allen Park, Michigan, United States"
1,AS SHOW,A-v-entus Eau de Parfum 3.3 oz 100ML Millesime...,Eau de Parfum,NaN,US $109.99,8.0,8 available / 48 sold,48.0,"May 23, 2024 23:07:49 PDT","Atlanta, Georgia, Canada"
2,Unbranded,HOGO BOSS cologne For Men 3.4 oz,Eau de Toilette,100.00,US $100.00,10.0,More than 10 available / 27 sold,27.0,"May 22, 2024 21:55:43 PDT","Dearborn, Michigan, United States"
3,Giorgio Armani,Acqua Di Gio by Giorgio Armani 6.7 Fl oz Eau D...,Eau de Toilette,NaN,US $44.99/ea,2.0,2 available / 159 sold,159.0,"May 24, 2024 03:30:43 PDT","Reinholds, Pennsylvania, United States"
4,Lattafa,Lattafa Men's Hayaati Al Maleky EDP Spray 3.4 ...,Fragrances,16.91,NaN,NaN,Limited quantity available / 156 sold,NaN,"May 24, 2024 07:56:25 PDT","Brooklyn, New York, United States"


In [5]:
df.tail()

,Item_Name,details,Catatory,price,priceWithCurrency,available,text,sold,lastUpdated,itemLocation
995,GUESS,Guess 1981 by Guess cologne for men EDT 3.3 / ...,Eau de Toilette,20.28,US $20.28/ea,45.0,"45 available / 1,613 sold",1613.0,"May 24, 2024 08:14:07 PDT","Dallas, Texas, United States"
996,Armaf,Club de Nuit Intense by Armaf cologne for men ...,Eau de Toilette,30.58,US $30.58,10.0,More than 10 available / 31 sold,31.0,"May 23, 2024 08:39:30 PDT",United States
997,Paco Rabanne,Invictus by Paco Rabanne for Men EDT Spray 3.4...,Eau de Toilette,39.99,US $39.99/ea,2.0,2 available / 305 sold,305.0,"May 23, 2024 15:27:18 PDT","Jamaica, New York, United States"
998,Lomani,"Lomani EDT Cologne 3.4 oz Men - Authentic, Bra...",Eau de Toilette,9.99,US $9.99/ea,2.0,2 available / 22 sold,22.0,"May 20, 2024 13:20:54 PDT","Lincoln Park, Michigan, United States"
999,Estee Lauder,Beyond Paradise by Estee Lauder for Men Cologn...,Cologne spray,17.49,US $17.49/ea,10.0,More than 10 available / 24 sold,24.0,"Feb 28, 2024 07:27:01 PST","Keyport, New Jersey, United States"


In [6]:
df.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Item_Name,999,246,Giorgio Armani,60,NaN,NaN,NaN,NaN,NaN,NaN,NaN
details,1000,966,Hawas for him Eau De Parfum By Rasasi 100ml 3....,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Catatory,997,63,Eau de Toilette,444,NaN,NaN,NaN,NaN,NaN,NaN,NaN
price,995.0,NaN,NaN,NaN,46.433467,35.551712,3.0,22.99,35.71,59.0,259.09
priceWithCurrency,999,684,US $29.99/ea,21,NaN,NaN,NaN,NaN,NaN,NaN,NaN
available,889.0,NaN,NaN,NaN,20.046119,61.547985,2.0,5.0,10.0,10.0,842.0
text,997,758,More than 10 available / 16 sold,8,NaN,NaN,NaN,NaN,NaN,NaN,NaN
sold,992.0,NaN,NaN,NaN,775.246976,3211.433298,1.0,14.0,49.5,324.75,54052.0
lastUpdated,947,933,"May 24, 2024 10:26:59 PDT",4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
itemLocation,1000,276,"Dallas, Texas, United States",126,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
column_names = df.columns

column_names

Index(['Item_Name', 'details', 'Catatory', 'price', 'priceWithCurrency',
       'available', 'text', 'sold', 'lastUpdated', 'itemLocation'],
      dtype='str')

In [11]:
column_names = [c.lower() for c in df.columns]

df.columns = column_names

df.columns

Index(['item_name', 'details', 'catatory', 'price', 'pricewithcurrency',
       'available', 'text', 'sold', 'lastupdated', 'itemlocation'],
      dtype='str')

In [12]:
# Renaming catatory to category
df = df.rename(columns={'catatory': 'category'})

# Understand the data

- Handling Missing Values and Placeholders.
- Handling Duplicates.


### Handling Missing Values and Placeholders


In [13]:
df.isna().sum()

item_name              1
details                0
category               3
price                  5
pricewithcurrency      1
available            111
text                   3
sold                   8
lastupdated           53
itemlocation           0
dtype: int64

In [14]:
total_missing_nums = df.isna().sum().sum()

print(f'Total missing numbers: {total_missing_nums}')

Total missing numbers: 185


In [15]:
percentage_isna = round((100* total_missing_nums/df.size), 3)

print(f'Percentage of Missing Numbers: {percentage_isna}%.')

Percentage of Missing Numbers: 1.85%.


1.85% of the data will be lost if each row having a missing value is dropped. That is a huge amount of data to be lost. Also, since some of the columns that have missing values are text, they cannot be imputed with 0s or median value. Available column has 111 missing values and really needs to be studied because generalizing and imputing it with 0s or median value would drastically alter the quality of the data.


In [16]:
# Handling missing values in item_name, category, text, pricewithcurrency

unknown_cols = ['item_name', 'category', 'text', 'pricewithcurrency']

for uc in unknown_cols:
    df[uc] = df[uc].fillna('Unknown')

df.isna().sum()

item_name              0
details                0
category               0
price                  5
pricewithcurrency      0
available            111
text                   0
sold                   8
lastupdated           53
itemlocation           0
dtype: int64

Handling Price. If price is not available and pricewithcurrency is avaialable, the price can be inferred from the pricewithcurrency


In [22]:
extract_price = (
    df['pricewithcurrency']
    .astype(str)
    .str.replace(',', '', regex=False)       
    .str.extract(r'(\d+\.?\d*)')[0]         
    .astype(float)
)

df['price'] = df['price'].fillna(extract_price)

In [23]:
df.isna().sum()

item_name              0
details                0
category               0
price                  0
pricewithcurrency      0
available            111
text                   0
sold                   8
lastupdated           53
itemlocation           0
dtype: int64

### SOLD TEXT


In [24]:
sold_text = (
    df['text']
    .str.extract(r'([\d,]+)\s+sold', flags=re.IGNORECASE)[0]
    .str.replace(',', '', regex=False)
    .astype(float)
)

In [25]:
mask = df['sold'].isna() & df['text'].notna() & (df['text'] != 'Unknown')
print(mask.sum())
print(df.loc[mask, ['item_name', 'sold', 'text']])

5
             item_name  sold                                   text
0                 Dior   NaN      More than 10 available / 116 sold
4              Lattafa   NaN  Limited quantity available / 156 sold
152            Dossier   NaN                 More than 10 available
792  Bath & Body Works   NaN                            4 available
797           Tom Ford   NaN                           10 available


In [27]:
# Handling sold
df['sold'] = df['sold'].fillna(sold_text)

# If the units sold cannot be gotten from the text
df['sold'] = df['sold'].fillna(df['sold'].median())

df.isna().sum()

item_name              0
details                0
category               0
price                  0
pricewithcurrency      0
available            111
text                   0
sold                   0
lastupdated           53
itemlocation           0
dtype: int64

### LAST UPDATED DATE


In [28]:
cleaned = df['lastupdated'].str.replace(r'\s+[A-Za-z]{2,4}$', '', regex=True)

df['lastupdated_dt'] = pd.to_datetime(cleaned, format='%b %d, %Y %H:%M:%S', errors='coerce')

In [29]:
df['lastupdated_missing'] = df['lastupdated_dt'].isna().astype(int)
print(df['lastupdated_missing'].sum())

53


In [30]:
print(df.groupby('lastupdated_missing')[['price', 'sold', 'available']].median())

                     price  sold  available
lastupdated_missing                        
0                    35.99  60.0       10.0
1                    28.99   6.0        9.0


In [31]:
median_date = df['lastupdated_dt'].median()
df['lastupdated_dt'] = df['lastupdated_dt'].fillna(median_date)

ref_date = df['lastupdated_dt'].max() + pd.Timedelta(days=1)
df['days_since_update'] = (ref_date - df['lastupdated_dt']).dt.days

In [37]:
print(df['days_since_update'].describe().round(1))

count    1000.0
mean        8.7
std        31.2
min         1.0
25%         1.0
50%         2.0
75%         6.0
max       582.0
Name: days_since_update, dtype: float64


In [38]:
df.isna().sum()

item_name                0
details                  0
category                 0
price                    0
pricewithcurrency        0
available              111
text                     0
sold                     0
lastupdated             53
itemlocation             0
lastupdated_dt           0
lastupdated_missing      0
days_since_update        0
dtype: int64

In [39]:
df = df.drop(columns=['lastupdated'])

In [40]:
df.isna().sum()

item_name                0
details                  0
category                 0
price                    0
pricewithcurrency        0
available              111
text                     0
sold                     0
itemlocation             0
lastupdated_dt           0
lastupdated_missing      0
days_since_update        0
dtype: int64

### Stock Availability


In [42]:
df['text'] = df['text'].fillna('Unknown')
stock_part = df['text'].str.split(' / ').str[0]
print('Distinct stock phrases:', stock_part.nunique())
print(stock_part.value_counts().head(30))

Distinct stock phrases: 95
text
More than 10 available        297
3 available                   109
5 available                    62
Limited quantity available     59
2 available                    55
Last One                       47
4 available                    46
8 available                    45
10 available                   44
6 available                    43
9 available                    36
7 available                    34
13 available                    7
12 available                    4
31 available                    4
33 available                    3
36 available                    3
Unknown                         3
16 available                    3
15 available                    3
47 available                    3
19 available                    3
96 available                    2
48 available                    2
226 available                   2
44 available                    2
84 available                    2
66 available                    2
95 available    

In [44]:
# 3. Classify each listing's stock status from the text column
t = df['text']
has = lambda phrase: t.str.contains(phrase, case=False)

status = pd.Series(np.select(
    [t.eq('Unknown'),
     has('More than 10 available'),
     has('Last One'),
     has('Limited quantity'),
     has('lots available')],
     
    ['No text',
     'More than 10 available',
     'Last One',
     'Limited quantity available',
     'More than N lots available'],
    default='Exact count, e.g. "3 available"'), index=df.index)

# 4. Count each status: all listings, then only those with stock missing
print('All listings:')
print(status.value_counts().to_string())
print('\nListings with available missing:')
print(status[df['available'].isna()].value_counts().to_string())

# 5. Censoring check: what is stored when the text says "More than 10 available"?
print('\nStored value when text says "More than 10 available":',
      df.loc[status.eq('More than 10 available'), 'available'].unique())
print('Listings with exactly "10 available":',
      ((df['available'] == 10) & status.ne('More than 10 available')).sum())

All listings:
Exact count, e.g. "3 available"    592
More than 10 available             297
Limited quantity available          59
Last One                            47
No text                              3
More than N lots available           2

Listings with available missing:
Limited quantity available    59
Last One                      47
No text                        3
More than N lots available     2

Stored value when text says "More than 10 available": [10.]
Listings with exactly "10 available": 44


In [45]:
last_one = df['available'].isna() & df['text'].str.contains('Last One', case=False, na=False)
df.loc[last_one, 'available'] = 1
print(last_one.sum(), df['available'].isna().sum())

47 64


In [49]:
df['available_limited'] = df['text'].str.contains('Limited quantity', case=False, na=False).astype(int)

df['available_10plus']  = df['text'].str.contains('More than 10 available', case=False, na=False).astype(int)
print(df[['available_limited', 'available_10plus']].sum())

available_limited     59
available_10plus     297
dtype: int64


In [50]:
df['available'] = df['available'].fillna(df['available'].median())

In [51]:
df.isna().sum()

item_name              0
details                0
category               0
price                  0
pricewithcurrency      0
available              0
text                   0
sold                   0
itemlocation           0
lastupdated_dt         0
lastupdated_missing    0
days_since_update      0
available_limited      0
available_10plus       0
dtype: int64

### Category


In [53]:
print(df['category'].nunique())

64


In [55]:
df['category'].value_counts().head(10)

category
Eau de Toilette    444
Eau de Parfum      285
Eau de Cologne      56
Perfume             50
Fragrances          32
Eau De Parfum       15
Parfum              13
Eau de Perfume      10
Eau De Toilette      9
Gift Sets            6
Name: count, dtype: int64

In [ ]:
print(df['category_clean'].value_counts().tail(30))

In [59]:
import numpy as np

df['category_clean'] = (
    df['category'].astype(str).str.lower()
    .str.replace(r'[~]', ' ', regex=True)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)

# Step 2: map to product-type groups (first match wins, so order matters)
c = df['category_clean']
has = lambda p: c.str.contains(p, regex=True, na=False)

conditions = [
    # mixed: names more than one product type
    (has(r'toil') & has(r'parf')) |
    (has(r'toil') & has(r'cologne')) |
    (has(r'parf') & has(r'cologne')),
    has(r'deodorant|\boil\b|roll on|freshener|body spray|splash'),   
    has(r'gift|\bset'),                                              
    has(r'toil|\bedt\b'),                                            
    has(r'eau de parf|eau de perf|\bedp\b'),                        
    has(r'cologne|\bedc\b'),                                         
    has(r'parfum|elixir|extrait'),                                   
    has(r'perfume|fragrance'),                                       
]

choices = ['mixed', 'other_product', 'gift_set', 'eau_de_toilette',
           'eau_de_parfum', 'eau_de_cologne', 'parfum', 'perfume_generic']

df['category_grouped'] = np.select(conditions, choices, default='other')

# Step 3: check the mixed group and the sizes before merging
print(df.loc[df['category_grouped'] == 'mixed', 'category_clean'].value_counts())
print(df['category_grouped'].value_counts(), '\n')

# Step 4: merge groups under 20 rows into 'other'
small = ['other_product', 'gift_set', 'mixed']

df['category_grouped'] = df['category_grouped'].replace(dict.fromkeys(small, 'other'))

# Step 5: final result
print(df['category_grouped'].value_counts())

category_clean
eau de toilette, eau de parfum, eau de parfum intense    1
eau de toilette, cologne spray                           1
eau de parfum/ eau de toilette                           1
Name: count, dtype: int64
category_grouped
eau_de_toilette    460
eau_de_parfum      314
perfume_generic     85
eau_de_cologne      70
parfum              29
other               19
other_product       14
gift_set             6
mixed                3
Name: count, dtype: int64 

category_grouped
eau_de_toilette    460
eau_de_parfum      314
perfume_generic     85
eau_de_cologne      70
other               42
parfum              29
Name: count, dtype: int64


### ITEM LOCATION


In [60]:
df['country'] = df['itemlocation'].astype(str).str.split(',').str[-1].str.strip()

print(df['country'].value_counts())

country
United States    912
Hong Kong         52
Canada            11
China             11
Taiwan             6
India              3
Poland             2
Israel             1
Brazil             1
Portugal           1
Name: count, dtype: int64


In [ ]:
region = np.select(
    [df['country'] == 'United States',
     df['country'].isin(['Hong Kong', 'China', 'Taiwan'])],
    ['united_states', 'east_asia'],
    default='other'
)
print(df.groupby(region)[['price', 'sold', 'available']].median())
print(pd.Series(region).value_counts())

In [61]:
us_words = r'united states|usa|\bus\b|california|texas|new york|newyork|new jersey|virginia|nevada|florida'
before_last = df['itemlocation'].astype(str).str.rsplit(',', n=1).str[0].str.lower()

ambiguous = (df['country'] != 'United States') & before_last.str.contains(us_words, regex=True, na=False)
print(ambiguous.sum())
print(df.loc[ambiguous, ['itemlocation', 'country']])

54
                                       itemlocation    country
13                     Dayton,New Jersey, Hong Kong  Hong Kong
28                          New York,USA, Hong Kong  Hong Kong
67                       Dayton, New Jersey, Canada     Canada
85                        USA,California, Hong Kong  Hong Kong
86                       New York, New York, Taiwan     Taiwan
92                  El Cajon, California, Hong Kong  Hong Kong
93   California,United States, Hong Kong, Hong Kong  Hong Kong
101                   Walnut, California, Hong Kong  Hong Kong
102                   Dayton, New Jersey, Hong Kong  Hong Kong
104                       Ottawa,California, Canada     Canada
138               New York,United States, Hong Kong  Hong Kong
167   Ontario, California, United States, Hong Kong  Hong Kong
219                   California USA Taiwan, Taiwan     Taiwan
228                       USA,California, Hong Kong  Hong Kong
238                 Houston, Texas, HongKong, China 

In [62]:
df['ships_from_us'] = (df['country'] == 'United States').astype(int)
print(df['ships_from_us'].value_counts())

ships_from_us
1    912
0     88
Name: count, dtype: int64


In [ ]:
alt = df['ships_from_us'].copy()
alt[ambiguous] = 1                 # what rule B would give
print(df['ships_from_us'].value_counts())
print(alt.value_counts())
print(df.groupby(alt)[['price', 'sold']].median())

In [ ]:
group = np.select(
    [ambiguous, df['ships_from_us'] == 1],
    ['ambiguous', 'us'],
    default='overseas_clear'
)
print(pd.Series(group).value_counts())
print(df.groupby(group)[['price', 'sold', 'available']].median())

In [ ]:
print(df['details'].str.contains(r'\d+\s*ml', case=False).mean())
print(df['details'].str.contains(r'\d+(\.\d+)?\s*(fl\.?\s*)?oz', case=False).mean())

In [ ]:
num = ['price', 'sold', 'available', 'days_since_update']
print(df[num].describe())
print(df[num].skew())
df[['lastupdated_missing','ships_from_us']].mean()

In [ ]:
df.columns

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

num   = ['price', 'sold', 'available', 'days_since_update']
flags = ['lastupdated_missing', 'available_limited', 'available_10plus', 'ships_from_us']

print(df[num].describe().round(2))
print(df[num].skew().round(2))
print(df[flags].mean().round(3))

corr = df[num + flags].corr(method='spearman')
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Spearman correlation of candidate features')
plt.tight_layout()
plt.show()

Spread and skew of numeric features and frequency of binary flags, used to decide which features to keep and which need a log transform.


In [ ]:
num   = ['price', 'sold', 'available', 'days_since_update']
flags = ['lastupdated_missing', 'available_limited', 'available_10plus', 'ships_from_us']

print(df[num].describe().round(2))
print(df[num].skew().round(2))
print(df[flags].mean().round(3))

In [ ]:
df.nlargest(10, 'sold')[['details', 'price', 'sold', 'text']]

In [ ]:
print(df.loc[61, ['details', 'category', 'category_grouped', 'price']])

In [ ]:
df.loc[(df['available'] == 10) & (df['available_10plus'] == 0), 'text'].head(10)

In [ ]:
mask = (df['available'] == 10) & (df['available_10plus'] == 0)
print(mask.sum())
print(df.loc[mask, 'text'].str.replace(r'\d[\d,]*', 'N', regex=True).value_counts())

In [ ]:
print(df.loc[mask & ~df['text'].str.contains('sold', na=False), ['details', 'sold', 'text']])

In [ ]:
num = ['price', 'sold', 'available', 'days_since_update']
df_log = df[num].apply(np.log1p)
print(df_log.skew().round(2))

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(10, 12))
for i, col in enumerate(num):
    axes[i, 0].hist(df[col], bins=50)
    axes[i, 0].set_title(f'{col} (original)')
    axes[i, 1].hist(df_log[col], bins=50)
    axes[i, 1].set_title(f'{col} (log1p)')
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.preprocessing import StandardScaler, RobustScaler

std = pd.DataFrame(StandardScaler().fit_transform(df_log), columns=num)
rob = pd.DataFrame(RobustScaler().fit_transform(df_log), columns=num)

print(std.describe().round(2))
print(rob.describe().round(2))

In [ ]:
cat_dummies = pd.get_dummies(df['category_grouped'], prefix='cat', dtype=int)
print(cat_dummies.sum())

In [ ]:
flags = ['ships_from_us', 'available_limited', 'lastupdated_missing']

X_a = pd.concat([std, cat_dummies], axis=1)
X_b = pd.concat([std, cat_dummies, df[flags].reset_index(drop=True)], axis=1)

print(X_a.shape, X_b.shape)

PCA on both feature matrices. Explained variance per component shows how many dimensions are needed to keep most of the information.


In [ ]:
from sklearn.decomposition import PCA

for name, X in [('X_a', X_a), ('X_b', X_b)]:
    pca = PCA().fit(X)
    r = pca.explained_variance_ratio_
    table = pd.DataFrame({'variance': r.round(3), 'cumulative': r.cumsum().round(3)},
                         index=[f'PC{i+1}' for i in range(len(r))])
    print(name); print(table); print()

In [ ]:
pca_a = PCA().fit(X_a)
r = pca_a.explained_variance_ratio_
x = range(1, len(r) + 1)

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(x, r, label='Per component')
ax.plot(x, r.cumsum(), marker='o', color='black', label='Cumulative')
ax.axhline(0.85, linestyle='--', color='grey', label='85%')
ax.set_xlabel('Principal component')
ax.set_ylabel('Explained variance ratio')
ax.set_title('PCA explained variance (feature set A)')
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
loadings = pd.DataFrame(pca_a.components_[:3].T, index=X_a.columns,
                        columns=['PC1', 'PC2', 'PC3']).round(2)
print(loadings)

Products projected onto the first two components, before clustering.


In [ ]:
pc = PCA(n_components=2).fit_transform(X_a)

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(pc[:, 0], pc[:, 1], s=10, alpha=0.5)
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
ax.set_title('Listings on the first two principal components (set A)')
plt.tight_layout()
plt.show()